# **From Large Families to Longer Lives: Global Relationship Between Fertility and Life Expectancy (1964–2013)**

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
df = pd.read_csv("gapminder.csv")
df.head()

# inspect data
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.isnull().sum())
print(df.describe())

# delete missing values
df = df.dropna(subset = ["lifeExp", "Fertility", "pop", "ID"])

(12200, 7)
Index(['Country', 'Year', 'lifeExp', 'pop', 'Fertility', 'Region', 'ID'], dtype='str')
Country          str
Year           int64
lifeExp      float64
pop          float64
Fertility    float64
Region           str
ID               str
dtype: object
Country         0
Year            0
lifeExp      2089
pop           394
Fertility    2100
Region          0
ID             50
dtype: int64
               Year       lifeExp           pop     Fertility
count  12200.000000  10111.000000  1.180600e+04  10100.000000
mean    1988.500000     64.078600  2.196105e+07      4.028719
std       14.431461     11.122779  9.593930e+07      2.013968
min     1964.000000      6.000000  5.000000e+01      0.836000
25%     1976.000000     56.282500  2.516718e+05      2.175750
50%     1988.500000     67.157000  3.497884e+06      3.632500
75%     2001.000000     72.484000  1.139999e+07      5.905250
max     2013.000000     83.580000  1.359368e+09      9.223000


## Main Research Question: How did the relationship between fertility and life expectancy evolve across countries and continents between 1964 and 2013, and to what extent did countries converge toward lower fertility and higher life expectancy?


This question focuses on both temporal change and structural change. It examines not only how fertility and life expectancy changed independently, but also how the relationship between the two evolved over time.
This section contains several basic analysis of the data, mean, standard deviation and regression, etc. The end of the section will be and **interactive bubble chart** that shows how countries progressed through the demographic transition over time.

### Average Life expectancy and Fertility of Each Year

In [3]:
# analyze by year
global_trend = (
    df.groupby("Year").agg(
        ave_lifeExp = ("lifeExp", "mean"),
        ave_Fertility = ("Fertility", "mean")
    )
    .reset_index()
)
global_trend.head()

# calculate the overall change of life expectancy and fertility from 1964 to 2013
start = global_trend.iloc[0]
end = global_trend.iloc[-1]

print("Life expectancy change by:",
      round(end["ave_lifeExp"] - start["ave_lifeExp"],2))
print("Fertility change by:",
      round(end["ave_Fertility"] - start["ave_Fertility"],2))

Life expectancy change by: 14.72
Fertility change by: -2.65


### Correlation Analysis

In [4]:
# Analyze the relationship between the two variables using the whole sample
overall_correlation = df["Fertility"].corr(df["lifeExp"])
print("Overall correlation:",round(overall_correlation,2))

Overall correlation: -0.83


In [18]:
# calculate the overall correlation coefficient of each year
corr_by_year = (
    df.groupby("Year")
    .apply(lambda x: x["Fertility"].corr(x["lifeExp"]))
    .reset_index(name = "Correlation")
)

fig = px.line(
    corr_by_year,
    x = "Year",
    y = "Correlation",
    title = "Correlation Between Fertility and Life Expectancy Over Time",
    markers = True,
    height = 500,
    width = 800
)

fig.add_hline(
    y = -0.70,
    line_dash = "dash",
    line_color = "black"
)

fig.show()

The figure shows a consistently strong negative correlation between fertility and life expectancy from 1964 to 2013. This means that countries with higher life expectancy generally tended to have lower fertility rates, while countries with lower life expectancy tended to have higher fertility rates. The relationship became stronger from the mid-1960s to the late 1980s, reaching its strongest level around the late 1980s and early 1990s, with a correlation close to -0.85. After the mid-1990s, the correlation became slightly weaker but remained strongly negative. Overall, the result supports the demographic transition theory: as countries experience improvements in health, survival, and living standards, fertility rates tend to decline.

### Country-wide differences (convergence)
Whether countries are converging to the same level of fertility and life expectancy.

In [19]:
dispersion = (
    df.groupby("Year")
    .agg(
        lifeExp_std = ("lifeExp", "std"),
        Fertility_std = ("Fertility", "std")
    )
    .reset_index()
)

dispersion.head(50)

fig1 = px.line(
    dispersion,
    x = "Year",
    y = "lifeExp_std",
    title = "Change of std of life expectancy",
    markers = True,
    height = 500,
    width = 800
)
fig1.show()

fig2 = px.line(
    dispersion,
    x = "Year",
    y = "Fertility_std",
    title = "Change of std of Fertility",
    markers = True,
    height = 500,
    width = 800
)
fig2.show()

The figures show that demographic differences across countries generally became smaller between 1964 and 2013. Fertility shows a **clear convergence pattern** after the late 1970s, as more countries moved toward lower fertility levels. Life expectancy also shows **long-term convergence**, although the process was **less stable** and included **temporary fluctuations**. Overall, the results suggest that countries became more similar in demographic outcomes over time, but fertility converged more smoothly than life expectancy.


### Top 10 Countries With the Largest Change

In [7]:
country_change = (
    df[df["Year"].isin([1964,2013])]
    .pivot(
        index = "Country",
        columns = "Year",
        values = ["lifeExp","Fertility"])
)

country_change["lifeExp_change"] = (
    country_change[("lifeExp", 2013)] -
    country_change[("lifeExp", 1964)]
)

country_change["fertility_change"] = (
    country_change[("Fertility", 2013)] -
    country_change[("Fertility", 1964)]
)

display(country_change.sort_values(
    "lifeExp_change",
    ascending=False
).head(10))

display(country_change.sort_values(
    "fertility_change",
    ascending=True
).head(10))

lifeExp         Fertility        lifeExp_change  \
Year              1964    2013      1964   2013                  
Country                                                          
Maldives        38.911  77.919     7.179  2.256         39.008   
Bhutan          33.827  68.294     6.670  2.232         34.467   
Timor-Leste     35.724  67.538     6.347  5.855         31.814   
Tunisia         44.903  75.873     7.107  2.008         30.970   
Oman            45.895  76.552     7.263  2.853         30.657   
Cambodia        41.900  71.916     6.909  2.861         30.016   
Nepal           40.071  68.410     5.997  2.300         28.339   
Western Sahara  39.880  67.764     6.562  2.363         27.884   
Saudi Arabia    47.781  75.479     7.257  2.644         27.698   
Afghanistan     33.639  60.947     7.671  4.900         27.308   

               fertility_change  
Year                             
Country                          
Maldives                 -4.923  
Bhutan                   -4.438  
Timor-Leste              -0.492  
Tunisia                  -5.099  
Oman                     -4.410  
Cambodia                 -4.048  
Nepal                    -3.697  
Western Sahara           -4.199  
Saudi Arabia             -4.613  
Afghanistan              -2.771

lifeExp         Fertility        lifeExp_change  \
Year                       1964    2013      1964   2013                  
Country                                                                   
Libya                 48.914000  75.325     7.676  2.356      26.411000   
Greenland             63.236535  71.500     7.270  2.077       8.263465   
Mongolia              51.771000  67.503     7.558  2.436      15.732000   
Bahrain               57.182000  76.608     7.192  2.075      19.426000   
Tunisia               44.903000  75.873     7.107  2.008      30.970000   
Costa Rica            63.818000  79.930     6.886  1.795      16.112000   
United Arab Emirates  56.328000  76.841     6.861  1.801      20.513000   
Qatar                 64.139000  78.369     6.985  2.019      14.230000   
Iran                  47.173000  74.048     6.868  1.920      26.875000   
Maldives              38.911000  77.919     7.179  2.256      39.008000   

                     fertility_change  
Year                                   
Country                                
Libya                          -5.320  
Greenland                      -5.193  
Mongolia                       -5.122  
Bahrain                        -5.117  
Tunisia                        -5.099  
Costa Rica                     -5.091  
United Arab Emirates           -5.060  
Qatar                          -4.966  
Iran                           -4.948  
Maldives                       -4.923

### Interactive Bubble Chart
Show how countries progressed through the demographic transition, categorized by region, in the coordinate system of Fertility as x-axis and Life Expectancy as y-axis.

From the chart we can see that bubbles of countries are moving to the left top of the coordinate system, which means that countries are transforming to higher life expectancy and lower fertility rate.

In [8]:
from bokeh.plotting import figure, output_file, show
from bokeh.models import ColumnDataSource, Slider, HoverTool, CustomJS
from bokeh.layouts import column

In [9]:
print(sorted(df["Region"].unique().tolist()))

initial_year = 1964

df["bubble_size"] = np.cbrt(df["pop"]) / 10 # to prevent extremely large bubble size due to large population

region_color = {
    "America": "blue",
    "East Asia & Pacific": "red",
    "Europe & Central Asia": "green",
    "Middle East & North Africa": "orange",
    "South Asia": "purple",
    "Sub-Saharan Africa": "yellow"
}

df["color"] = df["Region"].map(region_color)

df_year = df[df["Year"] == initial_year].copy()
source = ColumnDataSource(df_year)

image = figure(
    width = 800,
    height = 600,
    title = f"Life Expectancy vs Fertility({initial_year})",
    x_axis_label = "Fertility (children per woman)",
    y_axis_label = "Life Expectancy (years)",
    tools = "pan, wheel_zoom, box_zoom, reset, save"
)

# draw bubbles
image.scatter(
    x = "Fertility",
    y = "lifeExp",
    size = "bubble_size",
    color = "color",
    source = source,
    fill_alpha = 0.6,
    line_color = "black",
    line_alpha = 0.3,
    legend_field = "Region"
)


# add the hovering tool
hover = HoverTool(tooltips = [
    ("Country", "@Country"),
    ("Year", "@Year"),
    ("Life Expectancy", "@lifeExp{0.0}"),
    ("Population", "@pop{0.0}"),
    ("Fertility", "@Fertility{0.0}"),
    ("Region", "@Region")
])
image.add_tools(hover)

image.legend.location = "bottom_left"
# image.legend.click_policy = "hide"

slider = Slider(
    start = 1964,
    end = 2013,
    value = 1964,
    step = 1,
    title = "Year",
)

data_by_year = {
    str(year): df[df["Year"] == year].to_dict(orient = "list")
    for year in sorted(df["Year"].unique())
}

# define the callback code
callback = CustomJS(args=dict(
    source=source,
    slider = slider,
    data = data_by_year,
    plot = image),
    code = """
    const year = slider.value.toString();
    data = data[year];
    source.data = data;
    plot.title.text = "Life Expectancy vs. Fertility (" + year + ")";
"""
)
slider.js_on_change("value", callback)

image.title.text_font_size = "18pt"
image.title.align = "center"

layout = column(slider, image)
output_file("Main visualization.html")
show(layout)

['America', 'East Asia & Pacific', 'Europe & Central Asia', 'Middle East & North Africa', 'South Asia', 'Sub-Saharan Africa']


## Sub-Question 1: Which continent experienced the largest overall demographic transition between 1964 and 2013?

The fastest demographic transition is defined as the continent that experienced the largest decline in fertility, the largest increase in life expectancy, and the greatest overall movement in the fertility-life expectancy space between 1964 and 2013.
I use transition distance at here, combining the moving distance of the two variables together, D=((ΔF)*2+(ΔL)*2)*0.5. The larger the D, the larger the movement of the continent in fertility-life expectancy coordinate system.

In [10]:
# analyze by region and year
region_trend = (
    df.groupby(["Year","Region"]).agg(
        lifeExp = ("lifeExp", "mean"),
        Fertility = ("Fertility", "mean")
    )
    .reset_index()
)

pivot = region_trend.pivot(
    index = "Region",
    columns = "Year",
    values = ["Fertility","lifeExp"]
)

pivot.columns = [
    f"{variable}_{year}"
    for variable,year in pivot.columns
]
pivot = pivot.reset_index()

pivot["fertility_change"] = (
    pivot["Fertility_2013"] - pivot["Fertility_1964"]
)
pivot["lifeExp_change"] = (
    pivot["lifeExp_2013"] - pivot["lifeExp_1964"]
)

pivot["transition_distance"] = np.sqrt(
    pivot["fertility_change"]**2
    + pivot["lifeExp_change"]**2
)

pivot = pivot.sort_values(
    "transition_distance",
    ascending = True
)

summary = pivot[
    ["Region",
    "fertility_change",
    "lifeExp_change",
    "transition_distance"]
].round(2)

display(summary)

,Region,fertility_change,lifeExp_change,transition_distance
2,Europe & Central Asia,-1.53,8.27,8.41
0,America,-3.32,14.62,15.00
5,Sub-Saharan Africa,-1.99,15.37,15.49
1,East Asia & Pacific,-3.27,17.28,17.59
3,Middle East & North Africa,-4.25,21.14,21.56
4,South Asia,-3.75,25.31,25.59


In [11]:
# plotting
fig = px.bar(
    summary,
    x = "transition_distance",
    y = "Region",
    orientation = "h",
    text = "transition_distance",
    title = "Demographic Transition Distance by Continent (1964-2013)",
    labels = {
        "transition_distance": "Transition Distance",
        "Region": "Region",
    },
    hover_data = {
        "fertility_change": ":.2f",
        "lifeExp_change": ":.2f",
        "transition_distance": ":.2f",
    }
)

fig.update_traces(
    texttemplate = "%{text:.2f}",
    textposition = "outside"
)
fig.update_layout(height = 500, width = 800)
fig.show()

The results show that South Asia experienced the largest overall demographic transition between 1964 and 2013, with the largest transition distance of 25.59. This indicates that South Asia had the greatest combined movement in the fertility–life expectancy space, meaning that the region experienced a substantial decline in fertility together with a strong improvement in life expectancy. The Middle East and North Africa also shows a large transition distance, suggesting rapid demographic change, while Europe and Central Asia records the smallest distance. This is likely because many countries in Europe and Central Asia had already reached relatively low fertility and high life expectancy by 1964, leaving less room for further demographic movement.

## Sub-Question 2: When did countries experience their demographic transition?

At what point did each country cross the threshold from high fertility and low life expectancy to low fertility and high life expectancy, and how did the timing differ across continents?

Defining a threshold which shows the year the country complete the transition is necessary. The demographic transition year is defined as the first year in which a country reaches replacement-level fertility (total fertility rate ≤ 2.1 children per woman) and a high level of survival (life expectancy at birth ≥ 60 years, the UN identifies people over 60 as older people, source: https://emergency.unhcr.org/protection/persons-risk/older-persons#:~:text=An%20older%20person%20is%20defined%20by%20the%20United,status%20%28grandparents%29%2C%20physical%20appearance%2C%20or%20age-related%20health%20conditions.). Considering random fluctuations, both conditions should be remaining satisfied for at least five consecutive years.

In [12]:
def find_transition_year(group):
    condition = (
        (group["Fertility"] <= 2.1) &
        (group["lifeExp"] >= 60)
    )

    year = group.loc[condition, "Year"]

    if len(year) == 0:
        return np.nan

    return year.min()

transition = (
    df.groupby("Country")
      .apply(find_transition_year)
      .reset_index(name="Transition Year")
)

def decade_bin(year):
    if pd.isna(year):
        return "Not reached"
    elif year < 1970:
        return "1960s"
    elif year < 1980:
        return "1970s"
    elif year < 1990:
        return "1980s"
    elif year < 2000:
        return "1990s"
    elif year < 2010:
        return "2000s"
    else:
        return "2010s"

transition["Decades"] = (transition["Transition Year"].apply(decade_bin))

In [13]:
import pycountry

def get_iso3(country):
    try:
        return pycountry.countries.lookup(country).alpha_3
    except:
        return None

transition["ISO3"] = transition["Country"].apply(get_iso3)
print(transition)

                 Country  Transition Year      Decades ISO3
0            Afghanistan              NaN  Not reached  AFG
1                Albania           2003.0        2000s  ALB
2                Algeria              NaN  Not reached  DZA
3                 Angola              NaN  Not reached  AGO
4    Antigua and Barbuda           1981.0        1980s  ATG
..                   ...              ...          ...  ...
196   West Bank and Gaza              NaN  Not reached  NaN
197       Western Sahara              NaN  Not reached  ESH
198          Yemen, Rep.              NaN  Not reached  NaN
199               Zambia              NaN  Not reached  ZMB
200             Zimbabwe              NaN  Not reached  ZWE

[201 rows x 4 columns]


In [14]:
fig = px.choropleth(
    transition,
    locations = "ISO3",
    locationmode = "ISO-3",
    color = "Decades",
    hover_name = "Country",
    hover_data = {"Transition Year":":.0f"},
    category_orders = {
        "Decades": [
            "1960s",
            "1970s",
            "1980s",
            "1990s",
            "2000s",
            "2010s",
            "Not reached"
        ]
    },
    color_discrete_map={
    "1960s": "#08306b",
    "1970s": "#2171b5",
    "1980s": "#6baed6",
    "1990s": "#fd8d3c",
    "2000s": "#e6550d",
    "2010s": "#a63603",
    "Not reached": "lightgray"
    },
    title = "World Demographic Transition Year Distribution",
    height = 700,
    width = 1000,
)

fig.show()

Overall, the map suggests that demographic transition did not occur at the same time across the world. High-income regions such as **Europe, North America**, and **Oceania** generally reached the threshold earlier, while **Latin America and parts of Asia** completed the transition later. In contrast, many **African countries** and some countries in **South Asia** and the **Middle East** have not yet reached the threshold. This indicates that demographic transition is closely related to long-term socioeconomic development, including public health, education, urbanization, and fertility decline.

## Sub-question 3: What was the typical GNI per capita when countries completed the demographic transition?

Gross National Income (GNI) is the total income earned by a country's residents, including net income from abroad, reflecting the overall economic well-being of a nation. The World Bank also uses GNI per capita (current US$) to assign income level of countries.
GNI data source: https://data.worldbank.org/indicator/NY.GNP.PCAP.CD

In [15]:
gni = pd.read_csv("GNI per capita.csv", skiprows = 4)

gni = gni.melt(
    id_vars = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name = "Year",
    value_name = "GNI"
)
gni["Year"] = pd.to_numeric(gni["Year"], errors = "coerce")
gni["GNI"] = pd.to_numeric(gni["GNI"], errors = "coerce")

gni = gni.dropna(subset=["GNI"])
gni = gni[["Country Name", "Country Code", "Year", "GNI"]]

merged = pd.merge(
    transition,
    gni[["Country Code", "Year", "GNI"]],
    left_on=['ISO3', 'Transition Year'],
    right_on=['Country Code', 'Year'],
    how='left',
    validate='many_to_one'
)

merged["GNI at Transition"] = round(merged["GNI"],2)
merged = merged[["Country","ISO3","Transition Year","Decades","GNI at Transition"]]

print(merged.head(20))

                Country ISO3  Transition Year      Decades  GNI at Transition
0           Afghanistan  AFG              NaN  Not reached                NaN
1               Albania  ALB           2003.0        2000s             1700.0
2               Algeria  DZA              NaN  Not reached                NaN
3                Angola  AGO              NaN  Not reached                NaN
4   Antigua and Barbuda  ATG           1981.0        1980s             2340.0
5             Argentina  ARG              NaN  Not reached                NaN
6               Armenia  ARM           1995.0        1990s              460.0
7                 Aruba  ABW           1995.0        1990s            16340.0
8             Australia  AUS           1976.0        1970s             7880.0
9               Austria  AUT           1972.0        1970s             2750.0
10           Azerbaijan  AZE           2000.0        2000s              630.0
11              Bahamas  BHS           2000.0        2000s      

In [16]:
merged = merged.dropna(subset=["GNI at Transition"]).copy()

mean_gni = merged["GNI at Transition"].mean()
median_gni = merged["GNI at Transition"].median()

print(f"Mean GDP per capita at transition: {mean_gni:,.0f}")
print(f"Median GDP per capita at transition: {median_gni:,.0f}")

fig = px.histogram(
    merged,
    x="GNI at Transition",
    nbins=30,
    title="Typical GNI per Capita When Countries Completed the Demographic Transition",
    labels={
        "GNI at Transition": "GNI per Capita (constant US$)"
    },
    opacity=0.85
)

# add the average line
fig.add_vline(
    x=mean_gni,
    line_dash="dash",
    annotation_text=f"Mean: {mean_gni:,.0f}",
    annotation_position="top right"
)

# add the median line
fig.add_vline(
    x=median_gni,
    line_dash="solid",
    annotation_text=f"Median: {median_gni:,.0f}",
    annotation_position="top left"
)

fig.update_layout(
    xaxis_title="GDP per Capita at Transition (constant US$)",
    yaxis_title="Number of Countries",
    bargap=0.1,
    height = 600,
    width = 900,
)

fig.show()

Mean GDP per capita at transition: 7,562
Median GDP per capita at transition: 3,950


The distribution of GNI per capita at the time of demographic transition is highly right-skewed. The median income level is 4,520, far below the World Bank high-income threshold of 13,935 (source: https://datahelpdesk.worldbank.org/knowledgebase/articles/906519-world-bank-country-and-lending-groups). This indicates that most countries completed the demographic transition before reaching high-income status.
There are some possible reasons for this finding. Improvements in public health, education equality, child survival, urbanization, and access to contraception can reduce fertility and increase life expectancy even before a country becomes rich. Therefore, demographic transition can happen during the middle-income stage rather than only after countries become high-income economies.